In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

import lightgbm as lgb
from lightgbm import LGBMClassifier

import os
import json

import joblib

results = []
if os.path.exists("../reports/results.json"):
    try:
        with open("../reports/results.json") as f:
            results = json.load(f)
    except (json.JSONDecodeError, ValueError):
        results = []

def log_run(results, entry):
    results = [r for r in results if r["run_name"] != entry["run_name"]]
    results.append(entry)
    return results


df = pd.read_parquet('../data/loans_clean.parquet')

df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df["credit_history_months"] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df["annual_inc"].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

df_train = df[df['issue_year'].isin([2014, 2015])].copy()
df_val   = df[df["issue_year"] == 2016].copy()
df_test  = df[df['issue_year'] == 2017].copy()

y_train = df_train['default'].values
y_val   = df_val['default'].values

print(f"Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")

Train: 598,647  |  Val: 293,095  |  Test: 169,300


In [2]:
gain_rate = 0.10
loss_rate = 0.50

def calculate_profit(df_subset, approval_mask):
    """Total profit from a set of approval decisions."""
    approved = df_subset[approval_mask]
    if len(approved) == 0:
        return 0
    profit = np.where(
        approved["default"] == 0,
         gain_rate * approved["loan_amnt"],
        -loss_rate * approved["loan_amnt"]
    )
    return profit.sum()

In [3]:
df_train_fe = df_train.copy()
df_val_fe = df_val.copy()
df_test_fe = df_test.copy()

def prep_features(df):
    """Apply feature engineering decisions from EDA."""
    df = df.copy()
    
    redundant = [
        "fico_range_high",
        "funded_amnt",
        "funded_amnt_inv",
        "num_sats",
        "installment",
        "num_rev_tl_bal_gt_0",
    ]

    joint_cols = [c for c in df.columns if c.startswith("sec_app") or c.endswith("_joint")]

    high_cardinality = ["zip_code", "sub_grade"]

    split_cols = ["issue_year"]

    emp_map = {
        "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
        "5 years": 5, "6 years": 6, '7 years': 7, "8 years": 8, "9 years": 9,
        "10+ years": 10
    }
    df["emp_length"] = df["emp_length"].map(emp_map)
    
    cols_to_drop = redundant + joint_cols + high_cardinality + split_cols
    
    df = df.drop(
        columns=[c for c in cols_to_drop if c in df.columns]
    )
    
    return df



df_train_fe = prep_features(df_train_fe)
df_val_fe = prep_features(df_val_fe)
df_test_fe = prep_features(df_test_fe)

print(f"Features after prep: {df_train_fe.shape[1]}")
print(f"\nTrain shape: {df_train_fe.shape}")
print(f"Val shape:   {df_val_fe.shape}")

Features after prep: 80

Train shape: (598647, 80)
Val shape:   (293095, 80)


In [14]:
y_train = df_train_fe["default"].values
y_val = df_val_fe["default"].values

x_train = df_train_fe.drop(columns=["default"])
x_val = df_val_fe.drop(columns=['default'])

x_train['term'] = x_train['term'].str.extract(r'(\d+)').astype(int)
x_val['term']  = x_val['term'].str.extract(r'(\d+)').astype(int)

categorical_cols = x_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = x_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumeric columns: {len(numeric_cols)} features")

Categorical columns (7): ['grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type']

Numeric columns: 72 features


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# numeric pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
    ]
)

# categorical pipeline
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# both
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
    ]
) 


In [6]:
import time
import lightgbm as lgb
from lightgbm import LGBMClassifier

preprocessor.fit(x_train)

x_train_proc = preprocessor.transform(x_train)
x_val_proc = preprocessor.transform(x_val)

print(f"After preprocessing: train {x_train_proc.shape}, val {x_val_proc.shape}")

lgbm = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

print("Training LightGBM with early stopping...")

start = time.time()

lgbm.fit(
    x_train_proc,
    y_train,
    eval_set=[(x_val_proc, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=False
        )
    ]
)

train_time = time.time() - start

actual_iters = (
    lgbm.best_iteration_
    if hasattr(lgbm, "best_iteration_")
    else 1000
)

#Predictions
val_probs = lgbm.predict_proba(x_val_proc)[:, 1]

# Metrics
auc = roc_auc_score(y_val, val_probs)
pr_auc = average_precision_score(y_val, val_probs)
brier = brier_score_loss(y_val, val_probs)

# Profit optimization
thresholds = np.linspace(0.05, 0.55, 100)

profits = [
    calculate_profit(df_val, val_probs < t)
    for t in thresholds
]

best_idx = int(np.argmax(profits))

best_threshold = float(thresholds[best_idx])
best_profit = float(profits[best_idx])

approval_rate = float((val_probs < best_threshold).mean())

results = log_run(results, {
    "run_name": "lgbm_baseline",
    "model_type": "LGBMClassifier",
    "best_threshold": best_threshold,
    "auc": float(auc),
    "pr_auc": float(pr_auc),
    "brier": float(brier),
    "profit_at_threshold": float(best_profit),
    "approval_rate": float(approval_rate),
    "train_time_seconds": float(train_time),
    "actual_iterations": int(actual_iters),
})

print("\n=== LIGHTGBM (baseline) ===")
print(f"Train time:     {train_time:.1f}s")
print(f"Iterations:     {actual_iters} (max 1000)")
print(f"AUC:            {auc:.4f}")
print(f"PR-AUC:         {pr_auc:.4f}")
print(f"Brier:          {brier:.4f}")
print(f"Best threshold: {best_threshold:.4f}")
print(f"Approval rate:  {approval_rate:.2%}")
print(f"Profit:         ${best_profit:,.0f}")

rf_b  = next(r for r in results if r["run_name"] == "rf_baseline")
rf_nb = next(r for r in results if r["run_name"] == "rf_no_balance")

print("\n--- Full scoreboard ---")
print(f"RF baseline:    AUC {rf_b['auc']:.4f}, Brier {rf_b['brier']:.4f}, Profit ${rf_b['profit_at_threshold']/1e6:.1f}M")
print(f"RF no_balance:  AUC {rf_nb['auc']:.4f}, Brier {rf_nb['brier']:.4f}, Profit ${rf_nb['profit_at_threshold']/1e6:.1f}M")
print(f"LightGBM:       AUC {auc:.4f}, Brier {brier:.4f}, Profit ${best_profit/1e6:.1f}M")

After preprocessing: train (598647, 155), val (293095, 155)
Training LightGBM with early stopping...


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== LIGHTGBM (baseline) ===
Train time:     15.3s
Iterations:     501 (max 1000)
AUC:            0.7250
PR-AUC:         0.4411
Brier:          0.1585
Best threshold: 0.1359
Approval rate:  38.93%
Profit:         $63,714,818

--- Full scoreboard ---
RF baseline:    AUC 0.7159, Brier 0.2005, Profit $60.5M
RF no_balance:  AUC 0.7157, Brier 0.1609, Profit $60.5M
LightGBM:       AUC 0.7250, Brier 0.1585, Profit $63.7M


In [7]:
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    params = {
        'n_estimators': 1000,
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples':  trial.suggest_int('min_child_samples', 5, 100),
        'feature_fraction':   trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':   trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':       trial.suggest_int('bagging_freq', 0, 7),
        'lambda_l1':          trial.suggest_float('lambda_l1', 1e-3, 10, log=True),
        'lambda_l2':          trial.suggest_float('lambda_l2', 1e-3, 10, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    model = LGBMClassifier(**params)
    model.fit(
        x_train_proc, y_train,
        eval_set=[(x_val_proc, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    val_probs = model.predict_proba(x_val_proc)[:, 1]
    return roc_auc_score(y_val, val_probs)


print("Starting Optuna search (50 trials)...")

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)

start = time.time()
study.optimize(objective, n_trials=50, show_progress_bar=True)
search_time = time.time() - start

lgb_b = next(r for r in results if r["run_name"] == "lgbm_baseline")

print(f"\n=== OPTUNA SEARCH COMPLETE ===")
print(f"Total time: {search_time/60:.1f} min")
print(f"Trials completed: {len(study.trials)}")
print(f"Best AUC found: {study.best_value:.4f}")
print(f"Improvement over baseline ({lgb_b['auc']:.4f}): {study.best_value - lgb_b['auc']:+.4f}")

print(f"\nBest hyperparameters:")
for k, v in study.best_params.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-16 16:15:29,400] A new study created in memory with name: no-name-fb7de413-4bce-47c8-af17-6839a0c337e8


Starting Optuna search (50 trials)...


  0%|                                                                                           | 0/50 [00:00<?, ?it/s]/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 0. Best value: 0.726584:   2%|▉                                             | 1/50 [00:27<22:31, 27.59s/it]

[I 2026-06-16 16:15:57,011] Trial 0 finished with value: 0.7265843892205655 and parameters: {'learning_rate': 0.030710573677773714, 'num_leaves': 122, 'min_child_samples': 75, 'feature_fraction': 0.7993292420985183, 'bagging_fraction': 0.5780093202212182, 'bagging_freq': 1, 'lambda_l1': 0.0017073967431528124, 'lambda_l2': 2.915443189153755}. Best is trial 0 with value: 0.7265843892205655.


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 0. Best value: 0.726584:   4%|█▊                                            | 2/50 [00:41<15:30, 19.39s/it]

...

[I 2026-06-16 17:13:18,656] Trial 47 finished with value: 0.7272566009469168 and parameters: {'learning_rate': 0.015263293501167865, 'num_leaves': 86, 'min_child_samples': 63, 'feature_fraction': 0.9655789338036683, 'bagging_fraction': 0.9232088816109265, 'bagging_freq': 3, 'lambda_l1': 0.004495642530443567, 'lambda_l2': 9.76200503222446}. Best is trial 22 with value: 0.7279764909872772.


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 22. Best value: 0.727976:  98%|███████████████████████████████████████████ | 49/50 [58:39<01:16, 76.95s/it]

[I 2026-06-16 17:14:09,192] Trial 48 finished with value: 0.7268555399989455 and parameters: {'learning_rate': 0.019870999568465744, 'num_leaves': 101, 'min_child_samples': 55, 'feature_fraction': 0.6605895174560708, 'bagging_fraction': 0.678790302408587, 'bagging_freq': 4, 'lambda_l1': 0.009353306150737638, 'lambda_l2': 0.03721120144546231}. Best is trial 22 with value: 0.7279764909872772.


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Best trial: 22. Best value: 0.727976: 100%|████████████████████████████████████████████| 50/50 [59:58<00:00, 71.97s/it]

[I 2026-06-16 17:15:27,792] Trial 49 finished with value: 0.727013553481939 and parameters: {'learning_rate': 0.011255042978411609, 'num_leaves': 122, 'min_child_samples': 73, 'feature_fraction': 0.5474600677215373, 'bagging_fraction': 0.8689578703833853, 'bagging_freq': 5, 'lambda_l1': 0.020624645073325423, 'lambda_l2': 2.206097953070022}. Best is trial 22 with value: 0.7279764909872772.

=== OPTUNA SEARCH COMPLETE ===
Total time: 60.0 min
Trials completed: 50
Best AUC found: 0.7280
Improvement over baseline (0.7250): +0.0030

Best hyperparameters:
  learning_rate: 0.0156
  num_leaves: 90
  min_child_samples: 60
  feature_fraction: 0.5850
  bagging_fraction: 0.7121
  bagging_freq: 3
  lambda_l1: 0.0068
  lambda_l2: 9.4656


In [15]:
best_params = study.best_params.copy()

best_params["n_estimators"] = 1000
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbose"] = -1

lgbm_tuned = LGBMClassifier(**best_params)

print("Training final LightGBM with tuned hyperparameters...")

start = time.time()

lgbm_tuned.fit(
    x_train_proc,
    y_train,
    eval_set=[(x_val_proc, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=False
        )
    ]
)

train_time = time.time() - start

actual_iters = (
    lgbm_tuned.best_iteration_
    if hasattr(lgbm_tuned, "best_iteration_")
    else 1000
)


val_probs = lgbm_tuned.predict_proba(x_val_proc)[:, 1]

auc = roc_auc_score(y_val, val_probs)
pr_auc = average_precision_score(y_val, val_probs)
brier = brier_score_loss(y_val, val_probs)


thresholds = np.linspace(0.05, 0.55, 100)

profits = [
    calculate_profit(df_val, val_probs < t)
    for t in thresholds
]

best_idx = int(np.argmax(profits))

best_threshold = float(thresholds[best_idx])
best_profit = float(profits[best_idx])

approval_rate = float((val_probs < best_threshold).mean())

results = log_run(results, {
    "run_name": "lgbm_tuned",
    "model_type":  "LGBMClassifier",
    "best_threshold": best_threshold,
    "auc": float(auc),
    "pr_auc": float(pr_auc),
    "brier": float(brier),
    "profit_at_threshold": float(best_profit),
    "approval_rate": float(approval_rate),
    "train_time_seconds":float(train_time),
    "actual_iterations": int(actual_iters),
    **best_params
})

print("\n=== LIGHTGBM TUNED ===")
print(f"Train time:     {train_time:.1f}s")
print(f"Iterations:     {actual_iters}")
print(f"AUC:            {auc:.4f}")
print(f"PR-AUC:         {pr_auc:.4f}")
print(f"Brier:          {brier:.4f}")
print(f"Best threshold: {best_threshold:.4f}")
print(f"Approval rate:  {approval_rate:.2%}")
print(f"Profit:         ${best_profit:,.0f}")

lgb_b = next(r for r in results if r["run_name"] == "lgbm_baseline")

print("\n--- Comparison to LightGBM baseline ---")
print(f"LGBM baseline: AUC {lgb_b['auc']:.4f}, Brier {lgb_b['brier']:.4f}, Profit ${lgb_b['profit_at_threshold']/1e6:.1f}M")
print(f"LGBM tuned:    AUC {auc:.4f}, Brier {brier:.4f}, Profit ${best_profit/1e6:.1f}M")

Training final LightGBM with tuned hyperparameters...


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== LIGHTGBM TUNED ===
Train time:     61.3s
Iterations:     1000
AUC:            0.7280
PR-AUC:         0.4453
Brier:          0.1578
Best threshold: 0.1308
Approval rate:  36.90%
Profit:         $65,382,568

--- Comparison to LightGBM baseline ---
LGBM baseline: AUC 0.7250, Brier 0.1585, Profit $63.7M
LGBM tuned:    AUC 0.7280, Brier 0.1578, Profit $65.4M


In [16]:
pd.DataFrame(results).sort_values(
    "profit_at_threshold",
    ascending=False
)


,run_name,model_type,predicted_default_prob,auc,pr_auc,brier,profit_at_threshold,approval_rate,best_threshold,train_time_seconds,...,num_leaves,min_child_samples,feature_fraction,bagging_fraction,bagging_freq,lambda_l1,lambda_l2,random_state,n_jobs,verbose
7,lgbm_tuned,LGBMClassifier,NaN,0.727976,0.445261,0.157796,65382567.5,0.369000,0.130808,61.284705,...,90.0,60.0,0.585003,0.712064,3.0,0.00682,9.465602,42.0,-1.0,-1.0
6,lgbm_baseline,LGBMClassifier,NaN,0.724994,0.441141,0.158509,63714817.5,0.389338,0.135859,15.266326,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,logreg_baseline,LogisticRegression,NaN,0.715654,0.424605,0.219360,60676972.5,0.358256,0.378283,17.850462,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,rf_baseline,RandomForestClassifier,NaN,0.715918,0.427309,0.200513,60507642.5,0.362596,0.358081,41.250288,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,rf_no_balance,RandomForestClassifier,NaN,0.715690,0.427888,0.160892,60461642.5,0.358587,0.140909,40.416495,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,logreg_no_int_rate,LogisticRegression,NaN,0.714598,0.424545,0.216012,59901460.0,0.344527,0.368182,16.662840,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,baseline_grade_only,grade_lookup,NaN,0.681608,0.359212,0.166423,42848472.5,0.470684,0.125758,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,baseline_approve_all,constant,0.195381,0.500000,0.232832,0.180024,-206712975.0,1.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
with open("../reports/results.json", "w") as f:
    json.dump(results, f, indent=2)

In [19]:
from pathlib import Path
Path("../models").mkdir(parents=True, exist_ok=True)

preprocessor.fit(x_train)

joblib.dump(lgbm_tuned, "../models/lgbm_tuned.pkl")
joblib.dump(preprocessor, "../models/preprocessor.pkl")
print("Saved:", [p.name for p in Path("../models").glob("*.pkl")])

Saved: ['preprocessor.pkl', 'lgbm_tuned.pkl']
